<a href="https://colab.research.google.com/github/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/blob/main/telco_bronze_to_silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# telco_bronze_to_silver.ipynb
**Dự án:** Telco Customer Churn – Medallion (Bronze → Silver) với BigQuery + dbt

| Phần | Nội dung | Phụ trách |
|---|---|---|
| 1 | Khảo sát & profiling dataset (21 cột) | Phân tích dataset & Bronze |
| 2 | Nạp CSV vào `bronze_telco.customers_raw` | Phân tích dataset & Bronze |
| 3 | dbt staging / Silver (`stg_telco_customers`, `customers_cleaned`) | Thành viên dbt |

Quy tắc đặt tên: file gốc `telco_customer_churn.csv` · Bronze `bronze_telco.customers_raw` · Silver `silver_telco.customers_cleaned`.

Cell 1 — Cấu hình và xác thực Google Cloud

In [3]:

PROJECT_ID  = "btqt2-telco-dbt-bigquery"
BQ_LOCATION = "asia-southeast1"       # giữ cùng location cho Bronze và Silver
BRONZE_DATASET = "bronze_telco"
BRONZE_TABLE   = "customers_raw"
SOURCE_FILE    = "telco_customer_churn.csv"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print("Đã kết nối BigQuery:", bq.project)


Đã kết nối BigQuery: btqt2-telco-dbt-bigquery


Cell 2 — Kiểm tra kết nối và tạo Bronze/Silver datasets

In [4]:

import os

SILVER_DATASET = "silver_telco"
SILVER_TABLE = "customers_cleaned"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

!gcloud config set project {PROJECT_ID} --quiet

print("Project:", bq.project)
print(
    "Kiểm tra kết nối:",
    list(bq.query("SELECT 1 AS ok").result())[0].ok
)

for name, desc in [
    (
        BRONZE_DATASET,
        "Bronze layer - du lieu Telco Customer Churn goc"
    ),
    (
        SILVER_DATASET,
        "Silver layer - du lieu da lam sach bang dbt"
    ),
]:
    d = bigquery.Dataset(f"{PROJECT_ID}.{name}")
    d.location = BQ_LOCATION
    d.description = desc

    d = bq.create_dataset(d, exists_ok=True)
    print("Dataset:", d.full_dataset_id, "| Location:", d.location)

print(
    "Danh sach dataset:",
    [d.dataset_id for d in bq.list_datasets()]
)


[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Project: btqt2-telco-dbt-bigquery
Kiểm tra kết nối: 1
Dataset: btqt2-telco-dbt-bigquery:bronze_telco | Location: asia-southeast1
Dataset: btqt2-telco-dbt-bigquery:silver_telco | Location: asia-southeast1
Danh sach dataset: ['bronze_telco', 'silver_telco']


Cell 3 — Upload CSV và đọc dữ liệu *gốc*

In [5]:

import os
import pandas as pd
from urllib.request import urlretrieve

SOURCE_FILE = "telco_customer_churn.csv"

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "huyenpiano1711/BTQT2_Telco_dbt_BigQuery/"
    "main/telco_customer_churn.csv"
)

urlretrieve(DATA_URL, SOURCE_FILE)

df = pd.read_csv(
    SOURCE_FILE,
    dtype=str,
    keep_default_na=False
)

print("Nguồn dữ liệu:", DATA_URL)
print("Kích thước file:", os.path.getsize(SOURCE_FILE), "bytes")
print("Kích thước dataset:", df.shape)

display(df.head())


Nguồn dữ liệu: https://raw.githubusercontent.com/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/main/telco_customer_churn.csv
Kích thước file: 977501 bytes
Kích thước dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


Cell 4 — Nạp dữ liệu vào BigQuery Bronze

In [6]:

dataset_id = f"{PROJECT_ID}.{BRONZE_DATASET}"

# Schema tường minh: 21 cột STRING + metadata
schema = [bigquery.SchemaField(c, "STRING") for c in df.columns] + [
    bigquery.SchemaField("_ingested_at", "TIMESTAMP"),
    bigquery.SchemaField("_source_file", "STRING"),
]

bronze_df = df.copy()
bronze_df["_ingested_at"] = pd.Timestamp.now(tz="UTC")
bronze_df["_source_file"] = SOURCE_FILE

table_id = f"{dataset_id}.{BRONZE_TABLE}"
job = bq.load_table_from_dataframe(
    bronze_df, table_id,
    job_config=bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE"),  # chạy lại không bị nhân đôi
)
job.result()
print("Đã nạp", bq.get_table(table_id).num_rows, "dòng vào", table_id)


Đã nạp 7043 dòng vào btqt2-telco-dbt-bigquery.bronze_telco.customers_raw


In [7]:
# Thống kê các cột số
num = df[["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]].apply(pd.to_numeric, errors="coerce")
num.describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00,7032.00
mean,0.16,32.37,64.76,2283.30
std,0.37,24.56,30.09,2266.77
min,0.00,0.00,18.25,18.80
25%,0.00,9.00,35.50,401.45
50%,0.00,29.00,70.35,1397.48
75%,0.00,55.00,89.85,3794.74
max,1.00,72.00,118.75,8684.80


Cell 3b — Profiling chất lượng dữ liệu (minh chứng cho mục 2.2 của báo cáo)

In [8]:
feat = [c for c in df.columns if c != "customerID"]
tc_blank = df["TotalCharges"].str.strip().eq("")
tenure_num = pd.to_numeric(df["tenure"])
mc = pd.to_numeric(df["MonthlyCharges"]); tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
variance = ((tc.fillna(0) - tenure_num*mc).abs() / (tenure_num*mc).where(tenure_num*mc != 0)) > 0.2

print("isnull() toàn bộ 21 cột          :", int(df.isnull().sum().sum()))
print("Chuỗi rỗng ''                    :", int((df == "").sum().sum()))
print("TotalCharges chỉ chứa khoảng trắng:", int(tc_blank.sum()), "| trong đó tenure = 0:", int((tc_blank & (tenure_num == 0)).sum()))
print("Trùng cả 21 cột                  :", int(df.duplicated().sum()))
print("customerID trùng                 :", int(df["customerID"].duplicated().sum()))
print("customerID sai mẫu 4 số-5 chữ    :", int((~df["customerID"].str.fullmatch(r"\d{4}-[A-Z]{5}")).sum()))
print("Dòng trùng 20 cột đặc trưng      :", int(df.duplicated(feat, keep=False).sum()), "dòng /", int(df.duplicated(feat).sum()), "bản trùng")
print("Tenure ngoài [0,72]              :", int((~tenure_num.between(0, 72)).sum()))
print("MonthlyCharges <= 0              :", int((mc <= 0).sum()))
print("Dòng lệch > 20% so với tenure×MonthlyCharges:", int(variance.sum()))
print("Giá trị 'No phone service'       :", int((df["MultipleLines"] == "No phone service").sum()))
print("Giá trị 'No internet service'    :", int((df["OnlineSecurity"] == "No internet service").sum()), "(mỗi cột dịch vụ Internet)")


isnull() toàn bộ 21 cột          : 0
Chuỗi rỗng ''                    : 0
TotalCharges chỉ chứa khoảng trắng: 11 | trong đó tenure = 0: 11
Trùng cả 21 cột                  : 0
customerID trùng                 : 0
customerID sai mẫu 4 số-5 chữ    : 0
Dòng trùng 20 cột đặc trưng      : 42 dòng / 22 bản trùng
Tenure ngoài [0,72]              : 0
MonthlyCharges <= 0              : 0
Dòng lệch > 20% so với tenure×MonthlyCharges: 59
Giá trị 'No phone service'       : 682
Giá trị 'No internet service'    : 1526 (mỗi cột dịch vụ Internet)


Cell 5 — Đối chiếu dữ liệu CSV và Bronze

In [9]:

q = f'''
SELECT
  COUNT(*)                                        AS so_dong,
  COUNT(DISTINCT customerID)                      AS so_customer_id,
  COUNTIF(TRIM(TotalCharges) = '')                AS total_charges_trong,
  ROUND(SUM(CAST(MonthlyCharges AS NUMERIC)), 2)  AS tong_monthly_charges,
  COUNTIF(Churn = 'Yes')                          AS churn_yes,
  COUNTIF(Churn = 'No')                           AS churn_no
FROM `{table_id}`
'''
r = bq.query(q).to_dataframe().iloc[0]
bronze_tbl = bq.get_table(table_id)

csv_stats = {
    "so_dong":              len(df),
    "so_customer_id":       df["customerID"].nunique(),
    "total_charges_trong":  (df["TotalCharges"].str.strip() == "").sum(),
    "tong_monthly_charges": round(df["MonthlyCharges"].astype(float).sum(), 2),
    "churn_yes":            (df["Churn"] == "Yes").sum(),
    "churn_no":             (df["Churn"] == "No").sum(),
}
doi_chieu = pd.DataFrame({
    "Chỉ số": list(csv_stats) + ["so_cot_nghiep_vu"],
    "CSV":    [float(v) for v in csv_stats.values()] + [df.shape[1]],
    "Bronze": [float(r[k]) for k in csv_stats] + [len([f for f in bronze_tbl.schema if not f.name.startswith("_")])],
})
doi_chieu["Kết quả"] = (doi_chieu["CSV"].round(2) == doi_chieu["Bronze"].round(2)).map({True: "Khớp", False: "Lệch"})
doi_chieu


,Chỉ số,CSV,Bronze,Kết quả
0,so_dong,7043.0,7043.0,Khớp
1,so_customer_id,7043.0,7043.0,Khớp
2,total_charges_trong,11.0,11.0,Khớp
3,tong_monthly_charges,456116.6,456116.6,Khớp
4,churn_yes,1869.0,1869.0,Khớp
5,churn_no,5174.0,5174.0,Khớp
6,so_cot_nghiep_vu,21.0,21.0,Khớp


Cell 6 — Đối chiếu customerID và kiểm tra NULL

In [10]:

ids_bq  = set(bq.query(f"SELECT customerID FROM `{table_id}`").to_dataframe()["customerID"])
ids_csv = set(df["customerID"])
print("Có trong CSV nhưng thiếu ở Bronze :", len(ids_csv - ids_bq))
print("Có ở Bronze nhưng không có trong CSV:", len(ids_bq - ids_csv))

null_expr = " + ".join(f"COUNTIF({c} IS NULL)" for c in df.columns)
print("Số ô NULL trên 21 cột ở Bronze    :", list(bq.query(f"SELECT {null_expr} AS n FROM `{table_id}`").result())[0].n)


Có trong CSV nhưng thiếu ở Bronze : 0
Có ở Bronze nhưng không có trong CSV: 0
Số ô NULL trên 21 cột ở Bronze    : 0


Cell 7 — Hiển thị schema và dữ liệu mẫu

In [11]:

# Minh chứng: schema (21 cột STRING + 2 cột metadata) và 10 dòng mẫu của Bronze
display(pd.DataFrame([(f.name, f.field_type, f.mode) for f in bronze_tbl.schema], columns=["column", "type", "mode"]))
bq.query(f"SELECT * FROM `{table_id}` LIMIT 10").to_dataframe()


,column,type,mode
0,customerID,STRING,NULLABLE
1,gender,STRING,NULLABLE
2,SeniorCitizen,STRING,NULLABLE
3,Partner,STRING,NULLABLE
4,Dependents,STRING,NULLABLE
5,tenure,STRING,NULLABLE
6,PhoneService,STRING,NULLABLE
7,MultipleLines,STRING,NULLABLE
8,InternetService,STRING,NULLABLE
9,OnlineSecurity,STRING,NULLABLE


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,_ingested_at,_source_file
0,3473-XIIIT,Female,0,Yes,No,16,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,1534.75,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
1,8709-KRDVL,Female,0,No,No,33,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,3320.6,No,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
2,8397-MVTAZ,Male,0,Yes,No,34,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3480,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
3,1666-JXLKU,Female,0,No,No,37,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3810.55,No,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
4,2669-OIDSD,Female,0,Yes,No,48,Yes,No,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,4834,No,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
5,2632-UCGVD,Male,1,Yes,No,66,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,6871.9,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
6,9647-ERGBE,Female,0,Yes,Yes,30,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3046.15,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
7,8634-MPHTR,Male,1,Yes,No,47,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,4871.05,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
8,5515-AKOAJ,Female,0,No,No,54,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.1,5440.9,Yes,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv
9,9094-AZPHK,Female,0,No,No,15,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.15,1415,No,2026-10-19 05:44:41.504483+00:00,telco_customer_churn.csv


PHẦN 2 — dbt Core: Bronze → Staging → Silver
Phần 1 phía trên được giữ theo notebook gốc. Phần 2 được ghép từ dbt_stg_telco_customers.ipynb và các tệp SQL/YAML trong dự án gốc. Chạy lần lượt các cell từ trên xuống. Tên thực tế: btqt2-telco-dbt-bigquery, bronze_telco.customers_raw, silver_telco.stg_telco_customers, silver_telco.customers_cleaned.

Cell 8 — Cài dbt và chuẩn bị xác thực

In [12]:
%pip -q install dbt-core dbt-bigquery
import os, subprocess
from google.colab import auth
auth.authenticate_user()

# Đảm bảo PROJECT_ID được định nghĩa an toàn
PROJECT_ID = globals().get('PROJECT_ID', 'btqt2-telco-dbt-bigquery')
BQ_LOCATION = globals().get('BQ_LOCATION', 'asia-southeast1')

os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
# Không gán GOOGLE_APPLICATION_CREDENTIALS tới đường dẫn JSON không tồn tại.
# dbt BigQuery method: oauth sử dụng Application Default Credentials của Colab.
print('Project:', PROJECT_ID, '| Location:', BQ_LOCATION)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.7 M

Cell 9 — Tạo project dbt và cấu hình kết nối

In [13]:
from pathlib import Path

# Đảm bảo các biến cấu hình được định nghĩa kể cả khi kernel bị khởi động lại
PROJECT_ID = globals().get('PROJECT_ID', 'btqt2-telco-dbt-bigquery')
SILVER_DATASET = globals().get('SILVER_DATASET', 'silver_telco')
BQ_LOCATION = globals().get('BQ_LOCATION', 'asia-southeast1')

DBT_DIR = Path("/content/dbt_telco")
(DBT_DIR / "models" / "staging").mkdir(parents=True, exist_ok=True)
(DBT_DIR / "models" / "silver").mkdir(parents=True, exist_ok=True)
(DBT_DIR / "tests").mkdir(parents=True, exist_ok=True)

(DBT_DIR / "dbt_project.yml").write_text("""name: 'dbt_telco'
version: '1.0.0'
config-version: 2
profile: 'dbt_telco'
model-paths: ["models"]
clean-targets: ["target", "dbt_packages"]
models:
  dbt_telco:
    +materialized: table
    staging:
      +materialized: view
""", encoding="utf-8")

(DBT_DIR / "profiles.yml").write_text(f"""dbt_telco:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: {PROJECT_ID}
      dataset: {SILVER_DATASET}
      location: {BQ_LOCATION}
      threads: 4
      timeout_seconds: 300
      priority: interactive
""", encoding="utf-8")
print("Đã tạo project:", DBT_DIR)

Đã tạo project: /content/dbt_telco


Cell 10 — Khai báo Bronze source và staging model

In [14]:
source_yaml = f'''version: 2

sources:
  - name: bronze_telco
    database: {PROJECT_ID}
    schema: bronze_telco
    tables:
      - name: customers_raw
        description: "Raw customer churn data loaded from CSV into Bronze layer"'''
(DBT_DIR / 'models/staging/_sources.yml').write_text(source_yaml, encoding='utf-8')
staging_sql = "with source as (\n    select *\n    from {{ source('bronze_telco', 'customers_raw') }}\n),\n\nrenamed_and_casted as (\n    select\n        trim(customerID)                                    as customer_id,\n        trim(gender)                                        as gender,\n        cast(SeniorCitizen as int64)                        as senior_citizen,\n        trim(Partner)                                       as partner,\n        trim(Dependents)                                    as dependents,\n        cast(tenure as int64)                               as tenure,\n        trim(PhoneService)                                  as phone_service,\n        trim(MultipleLines)                                 as multiple_lines,\n        trim(InternetService)                               as internet_service,\n        trim(OnlineSecurity)                                as online_security,\n        trim(OnlineBackup)                                  as online_backup,\n        trim(DeviceProtection)                              as device_protection,\n        trim(TechSupport)                                   as tech_support,\n        trim(StreamingTV)                                   as streaming_tv,\n        trim(StreamingMovies)                               as streaming_movies,\n        trim(Contract)                                      as contract,\n        trim(PaperlessBilling)                              as paperless_billing,\n        trim(PaymentMethod)                                 as payment_method,\n        cast(MonthlyCharges as numeric)                     as monthly_charges,\n        safe_cast(\n            nullif(trim(TotalCharges), '') as numeric\n        )                                                   as total_charges,\n        trim(Churn)                                         as churn,\n        _ingested_at,\n        _source_file\n    from source\n)\n\nselect *\nfrom renamed_and_casted"
(DBT_DIR / 'models/staging/stg_telco_customers.sql').write_text(staging_sql, encoding='utf-8')
print('Đã tạo source và staging model')

Đã tạo source và staging model


Cell 11 — Tạo model Silver và các kiểm thử

Quy tắc Silver: giữ logic TotalCharges (điền 0 cho khách tenure = 0 kèm cờ `is_total_charges_imputed`); chuẩn hóa `No phone service` (multiple_lines) và `No internet service` (6 cột dịch vụ Internet) về `No`; chuyển các cột cờ sang BOOL; gắn cờ `is_charge_variance`. Đã thêm kiểm thử `accepted_values ['Yes','No']` cho 7 cột dịch vụ sau chuẩn hóa.

In [15]:
silver_sql = "with staging as (\n    select *\n    from {{ ref('stg_telco_customers') }}\n),\n\nnormalized as (\n    select\n        customer_id,\n        gender,\n        case\n            when senior_citizen = 1 then true\n            when senior_citizen = 0 then false\n            else null\n        end as senior_citizen,\n        case\n            when partner = 'Yes' then true\n            when partner = 'No' then false\n            else null\n        end as partner,\n        case\n            when dependents = 'Yes' then true\n            when dependents = 'No' then false\n            else null\n        end as dependents,\n        tenure,\n        case\n            when phone_service = 'Yes' then true\n            when phone_service = 'No' then false\n            else null\n        end as phone_service,\n        case\n            when multiple_lines = 'No phone service' then 'No'\n            else multiple_lines\n        end as multiple_lines,\n        internet_service,\n        case\n            when online_security = 'No internet service' then 'No'\n            else online_security\n        end as online_security,\n        case\n            when online_backup = 'No internet service' then 'No'\n            else online_backup\n        end as online_backup,\n        case\n            when device_protection = 'No internet service' then 'No'\n            else device_protection\n        end as device_protection,\n        case\n            when tech_support = 'No internet service' then 'No'\n            else tech_support\n        end as tech_support,\n        case\n            when streaming_tv = 'No internet service' then 'No'\n            else streaming_tv\n        end as streaming_tv,\n        case\n            when streaming_movies = 'No internet service' then 'No'\n            else streaming_movies\n        end as streaming_movies,\n        contract,\n        case\n            when paperless_billing = 'Yes' then true\n            when paperless_billing = 'No' then false\n            else null\n        end as paperless_billing,\n        payment_method,\n        monthly_charges,\n        total_charges,\n        case\n            when churn = 'Yes' then true\n            when churn = 'No' then false\n            else null\n        end as churn,\n        _ingested_at,\n        _source_file\n    from staging\n),\n\nimputed as (\n    select\n        *,\n        tenure = 0 as is_new_customer,\n        tenure = 0 and total_charges is null as is_total_charges_imputed,\n        case\n            when tenure = 0 and total_charges is null then cast(0 as numeric)\n            else total_charges\n        end as total_charges_cleaned\n    from normalized\n)\n\nselect\n    customer_id,\n    gender,\n    senior_citizen,\n    partner,\n    dependents,\n    tenure,\n    phone_service,\n    multiple_lines,\n    internet_service,\n    online_security,\n    online_backup,\n    device_protection,\n    tech_support,\n    streaming_tv,\n    streaming_movies,\n    contract,\n    paperless_billing,\n    payment_method,\n    monthly_charges,\n    total_charges_cleaned as total_charges,\n    churn,\n    is_new_customer,\n    is_total_charges_imputed,\n    coalesce(\n        safe_divide(\n            abs(total_charges_cleaned - tenure * monthly_charges),\n            tenure * monthly_charges\n        ) > 0.2,\n        false\n    ) as is_charge_variance,\n    _ingested_at,\n    _source_file\nfrom imputed"
(DBT_DIR / 'models/silver/customers_cleaned.sql').write_text(silver_sql, encoding='utf-8')
schema_yaml = "version: 2\n\nmodels:\n  - name: customers_cleaned\n    description: >\n      Cleaned Silver customer dataset. Standardizes service categories and\n      boolean fields, imputes TotalCharges for new customers, and flags\n      potentially material charge differences for review.\n    columns:\n      - name: customer_id\n        description: Unique customer identifier.\n        data_tests:\n          - unique\n          - not_null\n      - name: gender\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Female', 'Male']\n      - name: senior_citizen\n        data_tests:\n          - not_null\n      - name: partner\n        data_tests:\n          - not_null\n      - name: dependents\n        data_tests:\n          - not_null\n      - name: tenure\n        data_tests:\n          - not_null\n      - name: phone_service\n        data_tests:\n          - not_null\n      - name: internet_service\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['DSL', 'Fiber optic', 'No']\n      - name: multiple_lines\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: online_security\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: online_backup\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: device_protection\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: tech_support\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: streaming_tv\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: streaming_movies\n        description: Đã chuẩn hóa 'No phone service' / 'No internet service' về 'No'.\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Yes', 'No']\n      - name: contract\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: ['Month-to-month', 'One year', 'Two year']\n      - name: paperless_billing\n        data_tests:\n          - not_null\n      - name: payment_method\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values:\n                  - 'Bank transfer (automatic)'\n                  - 'Credit card (automatic)'\n                  - 'Electronic check'\n                  - 'Mailed check'\n      - name: monthly_charges\n        data_tests:\n          - not_null\n      - name: total_charges\n        data_tests:\n          - not_null\n      - name: churn\n        data_tests:\n          - not_null\n          - accepted_values:\n              arguments:\n                values: [true, false]\n                quote: false\n      - name: is_total_charges_imputed\n        data_tests:\n          - not_null\n      - name: is_charge_variance\n        data_tests:\n          - not_null"
(DBT_DIR / 'models/silver/schema.yml').write_text(schema_yaml, encoding='utf-8')
test_sqls = {'charge_variance_warning.sql': "{{ config(severity='warn') }}\n\nselect\n    customer_id,\n    tenure,\n    monthly_charges,\n    total_charges,\n    is_charge_variance\nfrom {{ ref('customers_cleaned') }}\nwhere is_charge_variance", 'monthly_charges_positive.sql': "select\n    customer_id,\n    monthly_charges\nfrom {{ ref('customers_cleaned') }}\nwhere monthly_charges <= 0", 'tenure_in_range.sql': "select\n    customer_id,\n    tenure\nfrom {{ ref('customers_cleaned') }}\nwhere tenure < 0\n   or tenure > 72"}
for filename, sql in test_sqls.items():
    (DBT_DIR / 'tests' / filename).write_text(sql, encoding='utf-8')
for path in sorted(DBT_DIR.rglob('*')):
    if path.is_file() and 'target' not in path.parts:
        print(path.relative_to(DBT_DIR))

dbt_project.yml
models/silver/customers_cleaned.sql
models/silver/schema.yml
models/staging/_sources.yml
models/staging/stg_telco_customers.sql
profiles.yml
tests/charge_variance_warning.sql
tests/monthly_charges_positive.sql
tests/tenure_in_range.sql


Cell 12 — Kiểm tra kết nối và chạy đầy đủ dbt

In [16]:
import subprocess
commands = [
    ["dbt", "debug", "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR)],
    ["dbt", "run",   "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR)],
    # dbt test trả exit code 1 khi có test FAIL; WARN (severity: warn) vẫn trả 0
    ["dbt", "test",  "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR)],
]
for cmd in commands:
    print("\n>>>", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout[-4000:])      # in log để làm minh chứng (PASS / WARN / ERROR)
    if result.stderr.strip():
        print(result.stderr[-1500:])
    if result.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit={result.returncode}): {' '.join(cmd)}")
print("\nĐã chạy xong dbt debug, run, test.")



>>> dbt debug --project-dir /content/dbt_telco --profiles-dir /content/dbt_telco
05:46:02  Running with dbt=1.12.6
05:46:02  dbt version: 1.12.6
05:46:02  python version: 3.13.16
05:46:02  python path: /usr/bin/python3
05:46:02  os info: Linux-6.6.122+-x86_64-with-glibc2.39
05:46:06  Using profiles dir at /content/dbt_telco
05:46:06  Using profiles.yml file at /content/dbt_telco/profiles.yml
05:46:06  Using dbt_project.yml file at /content/dbt_telco/dbt_project.yml
05:46:06  adapter type: bigquery
05:46:06  adapter version: 1.12.1
05:46:06  Configuration:
05:46:06    profiles.yml file [OK found and valid]
05:46:06    dbt_project.yml file [OK found and valid]
05:46:06  Required dependencies:
05:46:07   - git [OK found]

05:46:07  Connection:
05:46:07    method: oauth
05:46:07    database: btqt2-telco-dbt-bigquery
05:46:07    execution_project: btqt2-telco-dbt-bigquery
05:46:07    schema: silver_telco
05:46:07    location: asia-southeast1
05:46:07    priority: interactive
05:46:07    ma

Cell 13 — Kiểm tra bảng Silver trên BigQuery

In [17]:
# Đảm bảo các cấu hình và client được thiết lập an toàn
PROJECT_ID = globals().get('PROJECT_ID', 'btqt2-telco-dbt-bigquery')
SILVER_DATASET = globals().get('SILVER_DATASET', 'silver_telco')
SILVER_TABLE = globals().get('SILVER_TABLE', 'customers_cleaned')

from google.cloud import bigquery
bq = globals().get('bq') or bigquery.Client(project=PROJECT_ID)

silver_id = f"{PROJECT_ID}.{SILVER_DATASET}.{SILVER_TABLE}"
staging_id = f"{PROJECT_ID}.{SILVER_DATASET}.stg_telco_customers"

for t in [staging_id, silver_id]:
    table = bq.get_table(t)
    print(t, "| Loại:", table.table_type, "| Số dòng:", table.num_rows)

display(bq.query(f"""
SELECT customer_id, tenure, monthly_charges, total_charges,
       churn, is_new_customer, is_total_charges_imputed
FROM `{silver_id}` LIMIT 10
""").to_dataframe())

btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers | Loại: VIEW | Số dòng: 0
btqt2-telco-dbt-bigquery.silver_telco.customers_cleaned | Loại: TABLE | Số dòng: 7043


,customer_id,tenure,monthly_charges,total_charges,churn,is_new_customer,is_total_charges_imputed
0,3393-FMZPV,25,100.250000000,2387.750000000,True,False,False
1,4238-JSSWH,35,102.050000000,3452.550000000,False,False,False
2,0365-BZUWY,17,102.550000000,1742.500000000,False,False,False
3,2107-FBPTK,30,104.100000000,3121.100000000,False,False,False
4,6771-XWBDM,57,104.500000000,5921.350000000,True,False,False
5,9851-QXEEQ,41,104.700000000,4346.400000000,True,False,False
6,3419-SNJJD,65,105.250000000,6786.400000000,True,False,False
7,1400-MMYXY,3,105.900000000,334.650000000,True,False,False
8,2081-VEYEH,3,107.950000000,318.600000000,False,False,False
9,6179-GJPSO,43,108.150000000,4600.700000000,True,False,False


Cell 14 — Đối chiếu Bronze và Silver

In [18]:
# Đảm bảo các biến cấu hình và client được thiết lập an toàn
PROJECT_ID = globals().get('PROJECT_ID', 'btqt2-telco-dbt-bigquery')
BRONZE_DATASET = globals().get('BRONZE_DATASET', 'bronze_telco')
BRONZE_TABLE = globals().get('BRONZE_TABLE', 'customers_raw')

SILVER_DATASET = globals().get('SILVER_DATASET', 'silver_telco')
SILVER_TABLE = globals().get('SILVER_TABLE', 'customers_cleaned')

table_id = globals().get('table_id', f"{PROJECT_ID}.{BRONZE_DATASET}.{BRONZE_TABLE}")
silver_id = globals().get('silver_id', f"{PROJECT_ID}.{SILVER_DATASET}.{SILVER_TABLE}")

from google.cloud import bigquery
bq = globals().get('bq') or bigquery.Client(project=PROJECT_ID)

compare_sql = f"""
WITH bronze AS (
  SELECT COUNT(*) AS row_count,
         COUNT(DISTINCT customerID) AS unique_customers,
         COUNTIF(TRIM(TotalCharges) = '') AS blank_total_charges,
         COUNTIF(SAFE_CAST(tenure AS INT64) = 0) AS zero_tenure
  FROM `{table_id}`
), silver AS (
  SELECT COUNT(*) AS row_count,
         COUNT(DISTINCT customer_id) AS unique_customers,
         COUNTIF(total_charges IS NULL) AS null_total_charges,
         COUNTIF(is_total_charges_imputed) AS imputed_count,
         COUNTIF(is_new_customer) AS zero_tenure
  FROM `{silver_id}`
)
SELECT bronze.*, silver.row_count AS silver_rows,
       silver.unique_customers AS silver_unique_customers,
       silver.null_total_charges, silver.imputed_count,
       silver.zero_tenure AS silver_zero_tenure
FROM bronze CROSS JOIN silver
"""
display(bq.query(compare_sql).to_dataframe())

,row_count,unique_customers,blank_total_charges,zero_tenure,silver_rows,silver_unique_customers,null_total_charges,imputed_count,silver_zero_tenure
0,7043,7043,11,11,7043,7043,0,11,11
